# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [2]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
print(divipola.shape)
print(list(divipola.columns))
display(divipola.head())

filas_divipola = divipola.shape[0]

divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()
print(divipola["codigo_municipio"].head())

formato_ok = divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}")
print("Códigos con cinco cifras:", formato_ok.sum())
print("Códigos con otro formato:", (~formato_ok).sum())
display(divipola.loc[~formato_ok, ["Código Municipio", "codigo_municipio"]].head())

print("Únicos:", divipola["codigo_municipio"].is_unique)
print("Vacíos:", divipola["codigo_municipio"].eq("").sum())
print("Nulos:", divipola["codigo_municipio"].isna().sum())

col_tipo = "Tipo: Municipio / Isla / Área no municipalizada"
print(divipola[col_tipo].value_counts(dropna=False))

dimension = divipola[[
    "codigo_municipio",
    "Nombre Departamento",
    "Nombre Municipio",
    col_tipo,
]].copy()
dimension.columns = ["codigo_municipio", "departamento", "municipio", "tipo"]
display(dimension.head())
print(dimension.shape)

# Comprobar formato y unicidad del código.
assert len(dimension) == filas_divipola == 1122
assert dimension["codigo_municipio"].is_unique
assert dimension["codigo_municipio"].ne("").all()
assert dimension["codigo_municipio"].notna().all()
assert dimension["tipo"].nunique() == 3

(1122, 7)
['Código Departamento', 'Nombre Departamento', 'Código Municipio', 'Nombre Municipio', 'Tipo: Municipio / Isla / Área no municipalizada', 'longitud', 'Latitud']


,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"


0    05001
1    05002
2    05004
3    05021
4    05030
Name: codigo_municipio, dtype: object
Códigos con cinco cifras: 1122
Códigos con otro formato: 0


,Código Municipio,codigo_municipio


Únicos: True
Vacíos: 0
Nulos: 0
Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64


,codigo_municipio,departamento,municipio,tipo
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


(1122, 4)


**Interpretación del resultado:**

Se creó `codigo_municipio` a partir de `Código Municipio` de DIVIPOLA, conservándolo como texto para no perder el cero inicial. Los **1.122** códigos tienen exactamente cinco cifras, son únicos y no tienen vacíos ni nulos.

La columna de tipo distingue tres clases de entidad territorial: **1.103** municipios, **18** áreas no municipalizadas y **1** isla, que suman las 1.122 entidades. `dimension` conserva todas con cuatro columnas: código, departamento, municipio y tipo.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [3]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
filas_eva = eva.shape[0]
print(eva.shape)
display(eva[["Código Dane municipio", "Año"]].head())

texto = eva["Código Dane municipio"].str.strip()
corto_numerico = texto.str.fullmatch(r"[0-9]{1,4}")
eva["codigo_municipio"] = texto.where(~corto_numerico, texto.str.zfill(5))
print("Completados con ceros:", corto_numerico.sum())

valido = eva["codigo_municipio"].str.fullmatch(r"[0-9]{5}")
eva["codigo_revision"] = ~valido
print("Vacíos:", eva["codigo_municipio"].eq("").sum())
print("Mal formados (incluye vacíos):", eva["codigo_revision"].sum())
display(eva.loc[eva["codigo_revision"], ["Código Dane municipio", "codigo_municipio"]].drop_duplicates().head())

eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
print(eva["anio"].min(), eva["anio"].max(), eva["anio"].isna().sum())

combinaciones = eva[["codigo_municipio", "anio"]].drop_duplicates()
print(combinaciones.shape)

unidas = combinaciones.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    indicator=True,
    validate="many_to_one",
)
sin_correspondencia = (
    unidas.loc[unidas["_merge"].eq("left_only"), ["codigo_municipio", "anio"]]
    .sort_values(["codigo_municipio", "anio"])
    .reset_index(drop=True)
)
print("Evaluadas:", len(combinaciones), "| Sin correspondencia:", len(sin_correspondencia))
display(sin_correspondencia)

# Obtener sin_correspondencia con código municipal y año.
assert len(eva) == filas_eva
assert not sin_correspondencia["codigo_municipio"].isin(dimension["codigo_municipio"]).any()

(166732, 18)


,Código Dane municipio,Año
0,05001,2021
1,05001,2021
2,05001,2022
3,05001,2022
4,05001,2023


Completados con ceros: 0
Vacíos: 0
Mal formados (incluye vacíos): 0


,Código Dane municipio,codigo_municipio


2019 2025 0
(7691, 2)
Evaluadas: 7691 | Sin correspondencia: 0


,codigo_municipio,anio


**Interpretación del resultado:**

Se crearon `codigo_municipio` y `anio` a partir de `Código Dane municipio` y `Año` de EVA, conservando las columnas originales y las 166.732 filas. Ningún código requirió completar ceros, ninguno estaba vacío y ninguno estaba mal formado: los códigos de EVA ya tenían cinco cifras.

Se obtuvieron **7.691** combinaciones distintas de código y año. Al compararlas con `dimension` mediante un anti-join por `codigo_municipio`, **ninguna quedó sin correspondencia**, por lo que `sin_correspondencia` está vacío. Es decir, todos los municipios que aparecen en EVA existen en el corte 2024 de DIVIPOLA.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [4]:
# Desarrollo de la actividad 3
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")
print(eva[["area_ha", "produccion_t"]].isna().sum())

# Crear papa_2024 y apta.
papa_2024 = eva.loc[eva["Cultivo"].eq("Papa") & eva["anio"].eq(2024)].copy()
print("Filas papa_2024:", len(papa_2024))

apta = papa_2024.loc[
    papa_2024["area_ha"].gt(0) & papa_2024["produccion_t"].ge(0)
].copy()
print("Filas aptas:", len(apta))

excluidas = papa_2024.loc[~papa_2024.index.isin(apta.index)]
print("Filas excluidas del indicador:", len(excluidas))
display(excluidas[["Municipio", "Cultivo", "anio", "area_ha", "produccion_t"]].head())

# Mostrar las exclusiones del indicador y conciliar los conteos.
print("papa_2024:", len(papa_2024), "| aptas:", len(apta), "| excluidas:", len(excluidas))
assert len(papa_2024) == len(apta) + len(excluidas)
assert apta["area_ha"].gt(0).all() and apta["produccion_t"].ge(0).all()

area_ha         0
produccion_t    0
dtype: int64
Filas papa_2024: 831
Filas aptas: 779
Filas excluidas del indicador: 52


,Municipio,Cultivo,anio,area_ha,produccion_t
2602,Betulia,Papa,2024,0.0,0.0
23281,Aquitania,Papa,2024,0.0,0.0
23834,Betéitiva,Papa,2024,0.0,0.0
24094,Boyacá,Papa,2024,0.0,0.0
24367,Buenavista,Papa,2024,0.0,0.0


papa_2024: 831 | aptas: 779 | excluidas: 52


**Interpretación del resultado:**

Se convirtieron `Área cosechada` y `Producción` a `area_ha` y `produccion_t` con `pd.to_numeric(errors="coerce")`. No hubo valores ausentes en ninguna de las dos, por lo que no fue necesario reemplazar nada y no se usó el cero como sustituto de una medida faltante.

Al seleccionar `Cultivo == "Papa"` y el año 2024 se obtuvieron **831** filas en `papa_2024`. De ellas, **779** cumplen área positiva y producción no negativa y quedaron en `apta`, y **52** quedaron excluidas del indicador.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [5]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
rendimiento = (
    apta
    .groupby(
        ["codigo_municipio", "Cultivo", "Estado físico del cultivo", "anio"],
        as_index=False,
        dropna=False,
    )
    .agg(
        registros=("Cultivo", "size"),
        produccion_total_t=("produccion_t", "sum"),
        area_total_ha=("area_ha", "sum"),
    )
)
print(rendimiento.shape)
display(rendimiento.head())

rendimiento["rendimiento_t_ha"] = (
    rendimiento["produccion_total_t"] / rendimiento["area_total_ha"]
)
display(rendimiento.head())
print(rendimiento["rendimiento_t_ha"].describe())

# Comprobar denominadores y cantidad de registros agrupados.
assert rendimiento["area_total_ha"].gt(0).all()
assert rendimiento["registros"].sum() == len(apta)
assert rendimiento["rendimiento_t_ha"].notna().all()
print("Registros agrupados:", rendimiento["registros"].sum(), "= len(apta):", len(apta))

(292, 7)


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0
1,05002,Papa,En fresco,2024,2,1650.0,165.0
2,05004,Papa,En fresco,2024,4,15.0,5.0
3,05079,Papa,En fresco,2024,2,268.0,34.0
4,05086,Papa,En fresco,2024,3,10550.0,905.0


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459


count    292.000000
mean      17.978680
std        6.587110
min        3.000000
25%       14.363880
50%       18.000000
75%       21.724340
max       48.763699
Name: rendimiento_t_ha, dtype: float64
Registros agrupados: 779 = len(apta): 779


**Interpretación del resultado:**

Se agrupó `apta` por código municipal, cultivo, estado físico del cultivo y año, y se obtuvo `rendimiento` con **292 grupos**. Cada grupo tiene la cantidad de registros, la suma de producción y la suma de área cosechada. Los registros agrupados suman **779**, igual que `len(apta)`, así que ninguna fila se perdió al agrupar, y todos los grupos tienen área total positiva.

`rendimiento_t_ha` se calculó como suma de producción dividida entre suma de área, y no como promedio de los rendimientos de cada fila. Así, los registros con más área pesan más en el resultado, que es lo correcto para un indicador agregado. Los valores van de **3,0 a 48,8 t/ha**, con una mediana de **18,0 t/ha**.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [6]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
filas_antes = len(rendimiento)
print("Filas antes de la unión:", filas_antes)

rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True,
)
print("Filas después de la unión:", len(rendimiento_municipal))
display(rendimiento_municipal.head())

print(rendimiento_municipal["_merge"].value_counts())

sin_nombre = rendimiento_municipal.loc[
    rendimiento_municipal["_merge"].eq("left_only"),
    ["codigo_municipio", "Cultivo", "anio"],
]
print("Códigos sin nombre territorial:", len(sin_nombre))
display(sin_nombre)

# Verificar la conservación de filas y presentar la cobertura de la unión.
assert len(rendimiento_municipal) == filas_antes
assert rendimiento_municipal["_merge"].eq("both").all()
print("Filas conservadas:", filas_antes, "=", len(rendimiento_municipal))

Filas antes de la unión: 292
Filas después de la unión: 292


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_total_t,area_total_ha,rendimiento_t_ha,departamento,municipio,tipo,_merge
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both


_merge
both          292
left_only       0
right_only      0
Name: count, dtype: int64
Códigos sin nombre territorial: 0


,codigo_municipio,Cultivo,anio


Filas conservadas: 292 = 292


**Interpretación del resultado:**

Se unió `rendimiento` con `dimension` mediante un left merge por `codigo_municipio`, declarando `validate="many_to_one"` e incorporando el indicador `_merge`. El resultado se guardó en `rendimiento_municipal`, que incorpora el departamento, el municipio y el tipo de entidad territorial.

La cantidad de filas se conservó: **292** antes y **292** después de la unión. Los **292** grupos tienen correspondencia en la dimensión (`both`), y **0** códigos quedaron sin nombre territorial, por lo que el listado de excluidos se muestra vacío. La validación de cardinalidad no falló, lo que confirma que `codigo_municipio` es único en `dimension`.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [7]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
ruta_rend = OUT / "rendimiento_municipal.csv"
rendimiento_municipal.to_csv(ruta_rend, index=False)
print("Archivo generado:", ruta_rend)

ruta_sin = OUT / "codigos_sin_correspondencia.csv"
sin_correspondencia.to_csv(ruta_sin, index=False)
print("Archivo generado:", ruta_sin)

assert ruta_rend.is_file() and ruta_rend.parent == OUT
assert ruta_sin.is_file() and ruta_sin.parent == OUT

rend_leido = pd.read_csv(ruta_rend, dtype=str)
sin_leido = pd.read_csv(ruta_sin, dtype=str)

assert len(rend_leido) == len(rendimiento_municipal)
assert list(sin_leido.columns) == ["codigo_municipio", "anio"]
assert len(sin_leido) == 0

print("Rendimiento_municipal:", len(rend_leido), "filas")
print("Codigos_sin_correspondencia: ", list(sin_leido.columns))

Archivo generado: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/rendimiento_municipal.csv
Archivo generado: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/codigos_sin_correspondencia.csv
Rendimiento_municipal: 292 filas
Codigos_sin_correspondencia:  ['codigo_municipio', 'anio']


**Interpretación del resultado:**

Se exportaron `rendimiento_municipal` (292 filas) a `rendimiento_municipal.csv` y `sin_correspondencia` a `codigos_sin_correspondencia.csv`, ambos sin índice y dentro de `OUT`. El segundo archivo no tiene filas porque ningún código de EVA quedó sin correspondencia, pero conserva sus cabeceras (`codigo_municipio`, `anio`), de modo que sigue siendo un archivo válido y su estructura es reconocible.

El rendimiento de un grupo es la producción total entre el área total. Si se promediaran los rendimientos de cada fila, un registro de 1 hectarea pesaría igual que uno de 900 hectareas, y el resultado no representaría la producción real por hectárea del grupo. Dividir suma de producción entre suma de área pondera cada registro por su área, de modo que los registros grandes pesan más.

EVA cubre varios años (2019-2025) y DIVIPOLA es un corte único de 2024. Que un código de EVA exista en `dimension` indica una **coincidencia de códigos**, no una **equivalencia histórica de límites**: un municipio puede haber cambiado de límites, haberse creado o haber cambiado de código entre un año y otro, y el código seguiría coincidiendo sin que el territorio sea el mismo. Por eso, la ausencia de códigos sin correspondencia confirma que los códigos se pueden nombrar con el catálogo de 2024, pero no garantiza que el municipio de 2019 o 2020 tuviera los mismos límites.

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.